# TEKNOFEST–Trendyol Relevance v3

Bu notebook public leaderboard'a threshold uydurmaz; query-cold OOF, fold-local negatif üretimi, ham logit saklama ve meta cross-fit kalibrasyon üzerine kuruludur. `0.95+` bir hedef, garanti değildir.

## En olası beş problem

1. Eski validation dengeli sentetik pair'ler üzerinde ve tek fold'da; gerçek testte query başına medyan 100, p99 204, maksimum 3.680 aday var.
2. Positive-unlabeled veride yüksek model skorlu bilinmeyenleri hard-negative yapmak false-negative gürültüsünü büyütüyor.
3. Qwen3 reranker sequence-classifier değildir; resmi prompt sonunda `yes_logit - no_logit` skorlanmalıdır.
4. Train/test term ID'leri ayrık; query shift, calibration ve threshold ikinci bir cross-fit gerektiriyor.
5. Tek model ailesi, yüksek korelasyon ve yalnız binary submission saklamak private-LB riskini artırıyor.

## Nihai mimari ve deney sırası

A/B/C validation → family/model-consensus false-negative filtresi → bi-encoder hard mining → Qwen3 0.6B → BGE reranker → Türkçe BERT seed ensemble → seçilmiş pair'lerde 4B teacher → OOF calibration/stacking → OOF-onaylı query policy → doğrulanmış submission.

Yaklaşık VRAM: Qwen embedding inference 4–7 GB, Qwen reranker QLoRA 6–10 GB, BGE reranker LoRA 8–12 GB, Türkçe BERT 3–6 GB. Süre kararı 4.096 pair benchmark'ından verilir; 10 saat tahmini aşılırsa iş fold/model bazında ayrı commit'e bölünür. Ayrıntılı gerekçe ve komutlar `docs/PIPELINE_V3.md` içindedir.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import sys
import zipfile
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd

if Path('/content').exists() and not Path('/content/drive/MyDrive').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

def find_project_root() -> Path:
    """Locate the uploaded v3 source tree locally, in Colab, or under Kaggle inputs."""
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content'), Path('/kaggle/working')]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(path.parent.parent for path in kaggle_input.glob('*/src/trendyol_v3_pipeline.py'))
    for candidate in candidates:
        if (candidate / 'src' / 'trendyol_v3_pipeline.py').exists():
            return candidate.resolve()
    archives = []
    if Path('/content').exists():
        archives.extend(sorted(Path('/content').glob('*relevance*v3*.zip')))
    if kaggle_input.exists():
        archives.extend(sorted(kaggle_input.glob('*/*relevance*v3*.zip')))
    if archives:
        for archive in archives:
            base = Path('/content') if Path('/content').exists() else Path('/kaggle/working')
            extracted = base / 'trendyol_relevance_v3_code'
            if not (extracted / 'src' / 'trendyol_v3_pipeline.py').exists():
                extracted.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(archive) as bundle:
                    bundle.extractall(extracted)
            if (extracted / 'src' / 'trendyol_v3_pipeline.py').exists():
                return extracted.resolve()
    raise FileNotFoundError('V3 kod ağacı bulunamadı; bundle ZIP dosyasını Colab /content alanına yükleyin')

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / 'src'))

from trendyol_v3_core import discover_data_paths

def find_data_dir(root: Path) -> Path:
    """Find one directory containing all five competition CSV files."""
    candidates = [root / 'data', Path('/content'), Path('/content/data')]
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        candidates.extend([drive_root / 'trendyol_data', drive_root / 'Kaggle_Trendyol' / 'data', drive_root / 'Kaggle_Trendyol'])
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(path for path in kaggle_input.glob('*') if path.is_dir())
    valid = []
    for candidate in candidates:
        try:
            discover_data_paths(candidate)
            valid.append(candidate)
        except (FileNotFoundError, ValueError):
            continue
    if len(valid) != 1:
        raise RuntimeError(f'Tam CSV seti içeren tek veri klasörü bekleniyordu; bulunan={valid}')
    return valid[0]

DATA_DIR = find_data_dir(ROOT)
if Path('/content').exists():
    WORK_DIR = Path('/content/trendyol_v3')
elif Path('/kaggle/working').exists():
    WORK_DIR = Path('/kaggle/working/trendyol_v3')
else:
    WORK_DIR = ROOT / 'artifacts' / 'v3' / 'notebook_run'
WORK_DIR.mkdir(parents=True, exist_ok=True)
print({'root': str(ROOT), 'data': str(DATA_DIR), 'work': str(WORK_DIR)})

In [ ]:
def dependency_report() -> pd.DataFrame:
    """Report the packages required by the CPU and GPU stages."""
    packages = ['numpy', 'pandas', 'sklearn', 'scipy', 'pyarrow', 'torch', 'transformers', 'sentence_transformers', 'peft', 'faiss']
    rows = []
    for package in packages:
        try:
            module = __import__(package)
            rows.append({'package': package, 'installed': True, 'version': getattr(module, '__version__', 'installed')})
        except ImportError:
            rows.append({'package': package, 'installed': False, 'version': None})
    return pd.DataFrame(rows)

DEPENDENCIES = dependency_report()
display(DEPENDENCIES)
transformers_row = DEPENDENCIES[DEPENDENCIES.package.eq('transformers')]
if not transformers_row.empty and transformers_row.iloc[0].installed:
    major_minor = tuple(int(value) for value in str(transformers_row.iloc[0].version).split('.')[:2])
    if major_minor < (4, 51):
        raise RuntimeError('Qwen3 için transformers>=4.51.3 gerekli; requirements-gpu.txt kurulmalı ve Colab runtime yeniden başlatılmalı')

## 1–4. Audit, Validation A/B/C ve model-hazır fold artefaktları

DEBUG örneklemesi satır kesmez; bütün query gruplarını seçer. Böylece query başına aday ve pozitif sayıları smoke testte de anlamlı kalır. `DEBUG=False` tam veriye aynı kod yoluyla geçer. Tam semantic split için `EMBEDDING_BACKEND='qwen'` kullanılır; lexical backend yalnız CPU smoke test proxy'sidir.

In [ ]:
from trendyol_v3_pipeline import PipelineConfig, run_cpu_pipeline

DEBUG = True
RUN_CPU_PREPARATION = True
EMBEDDING_BACKEND = 'lexical' if DEBUG else 'qwen'
N_SPLITS = 3
SEED = 42

PIPELINE_CONFIG = PipelineConfig(
    debug=DEBUG,
    debug_items=5_000,
    debug_train_pairs=5_000,
    debug_submission_pairs=5_000,
    n_splits=N_SPLITS,
    semantic_clusters=48 if DEBUG else 192,
    embedding_backend=EMBEDDING_BACKEND,
    negative_ratio=2,
    curriculum_epochs=(0, 2),
    retrieval_top_k=300,
    item_policy='audit',
    seed=SEED,
)
PREP_DIR = WORK_DIR / ('debug_run' if DEBUG else 'full_run')
if RUN_CPU_PREPARATION:
    PREPARATION = run_cpu_pipeline(DATA_DIR, PREP_DIR, PIPELINE_CONFIG)
    print({'pipeline_hash': PREPARATION['pipeline_hash'], 'output': str(PREP_DIR)})
else:
    print('CPU preparation kapalı; mevcut PREP_DIR artefaktları kullanılacak:', PREP_DIR)
PIPELINE_RUN_HASH = json.loads((PREP_DIR / 'pipeline_run.json').read_text())['pipeline_hash']
MODEL_RUN_DIR = WORK_DIR / 'models' / PIPELINE_RUN_HASH[:12]
MODEL_RUN_DIR.mkdir(parents=True, exist_ok=True)
print({'pipeline_hash': PIPELINE_RUN_HASH, 'model_run_dir': str(MODEL_RUN_DIR)})

In [ ]:
AUDIT = json.loads((PREP_DIR / 'audit' / 'schema_audit.json').read_text())
print('Train:', AUDIT['train'])
print('Submission:', AUDIT['submission'])
print('Overlap:', AUDIT['overlap'])
display(pd.read_csv(PREP_DIR / 'validation' / 'adversarial_top_features.csv').head(30))
display(pd.read_csv(PREP_DIR / 'validation' / 'group_item_family_overlap.csv'))
display(pd.DataFrame(json.loads((PREP_DIR / 'folds' / 'preparation_summary.json').read_text())['folds']).T)

## 5–7. Bi-encoder ve false-negative güvenlikli hard mining

Qwen3-Embedding query tarafında İngilizce instruction, product tarafında instruction olmadan last-token pooling kullanır. Multi-positive InfoNCE aynı query'nin bütün bilinen pozitiflerini pozitif maskede tutar. 256d FAISS araması sonrası known positives çıkarılır; miner çıktısı fold scope hash'iyle doğrulanmadan negatif üretimine giremez.

In [ ]:
from trendyol_v3_biencoder import (
    BiEncoderAdapter, BiEncoderConfig, encode_to_memmap_cache,
    mine_faiss_hard_negatives, train_biencoder_with_oom_fallback,
)
from trendyol_v3_core import AuditConfig, load_data_bundle
from trendyol_v3_mining import (
    CatalogIndex, NegativeMiningConfig, build_fold_scope, generate_fold_negatives,
)
from trendyol_v3_pipeline import prepare_fold_training_frame
from trendyol_v3_reranker import TrainConfig, build_selected_item_view
from trendyol_v3_validation import SplitConfig, SplitManifest

RUN_BIENCODER = False
BIENCODER_CONFIG = BiEncoderConfig(
    model_name='Qwen/Qwen3-Embedding-0.6B',
    max_length=256, output_dimension=1024,
    matryoshka_dimensions=(1024, 512, 256),
    temperature=0.05, use_lora=True, use_qlora=True,
)
BI_TRAIN_CONFIG = TrainConfig(
    epochs=1, micro_batch_size=2, effective_batch_size=16,
    queries_per_batch=2, positives_per_query=2, negatives_per_positive=3,
    learning_rate=2e-5, label_smoothing=0.0, pairwise_weight=0.0, seed=SEED,
)
if RUN_BIENCODER:
    bi_bundle = load_data_bundle(DATA_DIR, AuditConfig(
        debug=DEBUG, debug_items=5_000, debug_train_pairs=5_000,
        debug_submission_pairs=5_000, seed=SEED,
    ))
    manifest_meta = json.loads((PREP_DIR / 'validation' / 'group_manifest.json').read_text())
    group_manifest = SplitManifest(
        pd.read_parquet(PREP_DIR / 'validation' / 'group_manifest.parquet'),
        SplitConfig(**manifest_meta['config']), manifest_meta['source_hash'],
    )
    item_view = build_selected_item_view(bi_bundle.items, 'long')
    query_vocabulary = {
        token for query in bi_bundle.terms.query.fillna('')
        for token in str(query).split() if len(token) >= 3
    }
    bi_catalog = CatalogIndex.build(bi_bundle.items, query_vocabulary)
    for fold in range(N_SPLITS):
        train_path = PREP_DIR / 'folds' / f'fold_{fold}' / 'epoch_2_train.parquet'
        output = MODEL_RUN_DIR / f'biencoder_fold{fold}'
        model_path = train_biencoder_with_oom_fallback(
            pd.read_parquet(train_path), output, BIENCODER_CONFIG, BI_TRAIN_CONFIG, device='cuda'
        )
        scope = build_fold_scope(group_manifest, bi_bundle.training_pairs, fold)
        adapter = BiEncoderAdapter(BIENCODER_CONFIG, device='cuda', checkpoint=model_path, trainable=False)
        query_frame = bi_bundle.terms[bi_bundle.terms.term_id.astype(str).isin(scope.train_term_ids)][['term_id', 'query']]
        cache_dir = output / 'embedding_cache'
        query_embeddings, query_map = encode_to_memmap_cache(
            query_frame, adapter, cache_dir, id_column='term_id', text_column='query',
            role='query', batch_size=16,
        )
        item_embeddings, item_map = encode_to_memmap_cache(
            item_view, adapter, cache_dir, id_column='item_id', text_column='product_text',
            role='document', batch_size=16,
        )
        mined = mine_faiss_hard_negatives(
            query_embeddings, query_map, item_embeddings, item_map, bi_bundle.training_pairs,
            output / 'bi_encoder_hard_negatives.parquet', search_dimension=256,
            top_k=500, output_per_query=100, miner_model_hash=f'{PIPELINE_RUN_HASH}:fold{fold}',
            miner_train_term_hash=scope.train_term_hash,
        )
        remine_config = NegativeMiningConfig(negative_ratio=2, seed=SEED)
        negatives, suspicious = generate_fold_negatives(
            scope, bi_bundle.training_pairs, bi_bundle.terms, bi_catalog, remine_config,
            epoch=2, external_candidates=mined, miner_model_hash=f'{PIPELINE_RUN_HASH}:fold{fold}',
        )
        remine_train = prepare_fold_training_frame(
            scope, bi_bundle.training_pairs, negatives, bi_bundle.terms, bi_bundle.items,
        )
        fold_dir = PREP_DIR / 'folds' / f'fold_{fold}'
        negatives.to_parquet(fold_dir / 'epoch_2_biencoder_negatives.parquet', index=False)
        suspicious.to_parquet(fold_dir / 'epoch_2_biencoder_suspicious.parquet', index=False)
        remine_train.to_parquet(fold_dir / 'epoch_2_biencoder_train.parquet', index=False)
        print({'fold': fold, 'model': str(model_path), 'mined_rows': len(mined), 'train_rows': len(remine_train)})
else:
    print('Bi-encoder eğitimi kapalı. CPU fold artefaktları hazır; RUN_BIENCODER=True ile GPU aşaması başlar.')

## 8. Qwen3, BGE ve Türkçe BERT query-aware eğitimi

Outer validation checkpoint seçimi için kullanılmaz. `LOCKED_EPOCHS` ayrı inner pilotta kilitlenir; her outer fold aynı epoch ile eğitilir ve oracle validation bir kez skorlanır. Qwen causal yes/no; BGE pretrained one-logit head; Türkçe BERT ise açık `initialize_sequence_head=True` yoluyla yeni tek-logit head kullanır.

In [ ]:
from trendyol_v3_reranker import (
    InferenceConfig, RerankerAdapter, RerankerConfig,
    score_oof_frame, train_with_oom_fallback,
)

import torch

RUN_RERANKERS = False
LOCKED_EPOCHS = 2
TRAIN_STAGE_FILE = 'epoch_2_biencoder_train.parquet' if RUN_BIENCODER else 'epoch_2_train.parquet'
INFERENCE_CONFIG = InferenceConfig(batch_size=32, shard_size=50_000, device='cuda', seed=SEED)
MODEL_SPECS = {
    'qwen': {
        'config': RerankerConfig(
            model_name='Qwen/Qwen3-Reranker-0.6B', architecture='qwen_causal',
            max_length=256, product_view='long', use_lora=True, use_qlora=True,
        ),
        'seeds': (42,), 'multi_sample_dropout': 1, 'fgm': False, 'swa': False,
    },
    'bge': {
        'config': RerankerConfig(
            model_name='BAAI/bge-reranker-v2-m3', architecture='sequence_classifier',
            max_length=256, product_view='long', use_lora=True, use_qlora=False,
        ),
        'seeds': (42,), 'multi_sample_dropout': 1, 'fgm': False, 'swa': False,
    },
    'trbert': {
        'config': RerankerConfig(
            model_name='dbmdz/bert-base-turkish-cased', architecture='sequence_classifier',
            initialize_sequence_head=True, max_length=192, product_view='long',
            use_lora=False, use_qlora=False,
            dtype='bf16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'fp32',
        ),
        'seeds': (42,) if DEBUG else (41, 42, 43),
        'multi_sample_dropout': 5, 'fgm': True, 'swa': True,
    },
}

def train_and_score_model_family(model_key: str, spec: Mapping[str, Any]) -> list[Path]:
    """Train fixed-epoch outer folds/seeds and persist complete OOF logits."""
    outputs = []
    for seed in spec['seeds']:
        for fold in range(N_SPLITS):
            fold_dir = PREP_DIR / 'folds' / f'fold_{fold}'
            train_frame = pd.read_parquet(fold_dir / TRAIN_STAGE_FILE)
            valid_frame = pd.read_parquet(fold_dir / 'validation_oracle.parquet')
            train_config = TrainConfig(
                epochs=LOCKED_EPOCHS, micro_batch_size=2, effective_batch_size=32,
                queries_per_batch=2, positives_per_query=2, negatives_per_positive=3,
                learning_rate=2e-5, label_smoothing=0.02, pairwise_weight=0.20,
                listwise_weight=0.0, distillation_weight=0.25, auxiliary_weight=0.10,
                multi_sample_dropout=spec['multi_sample_dropout'],
                use_fgm=spec['fgm'], use_swa=spec['swa'], seed=seed,
            )
            run_dir = MODEL_RUN_DIR / model_key / f'seed_{seed}' / f'fold_{fold}'
            checkpoint = train_with_oom_fallback(
                train_frame, run_dir, spec['config'], train_config, device='cuda'
            )
            adapter = RerankerAdapter(spec['config'], device='cuda', checkpoint=checkpoint, trainable=False)
            oof, metrics = score_oof_frame(
                valid_frame, adapter, INFERENCE_CONFIG, fold=fold,
                model_key=f'{model_key}_seed{seed}', seed=seed,
            )
            oof_path = run_dir / 'oof.parquet'
            oof.to_parquet(oof_path, index=False)
            (run_dir / 'oof_metrics.json').write_text(json.dumps(metrics, indent=2))
            outputs.append(oof_path)
    return outputs

OOF_PATHS = {}
if RUN_RERANKERS:
    for model_key, spec in MODEL_SPECS.items():
        OOF_PATHS[model_key] = train_and_score_model_family(model_key, spec)
else:
    print('Reranker eğitimi kapalı. RUN_RERANKERS=True tek-GPU fixed-epoch OOF eğitimini başlatır.')

## 9. 4B teacher distillation

Teacher bütün 3,36 milyon test pair'inde çalışmaz. Bütün pozitifler ile hard, uncertain, disagreement ve test-like pair'lerden en fazla 250 bin satır seçilir. Çıktı binary etikete çevrilmez; `teacher_logit` ve `teacher_probability` öğrenci frame'ine merge edilir.

In [ ]:
from trendyol_v3_ensemble import select_teacher_pairs
from trendyol_v3_reranker import score_frame_adaptive

RUN_TEACHER = False
if RUN_TEACHER:
    source_frame = pd.read_parquet(PREP_DIR / 'folds' / 'fold_0' / 'epoch_2_train.parquet')
    teacher_pairs = select_teacher_pairs(source_frame, max_rows=250_000, seed=SEED)
    teacher_config = RerankerConfig(
        model_name='Qwen/Qwen3-Reranker-4B', architecture='qwen_causal',
        max_length=256, product_view='long', use_lora=True, use_qlora=True,
    )
    teacher = RerankerAdapter(teacher_config, device='cuda', trainable=False)
    teacher_logits, used_batch = score_frame_adaptive(teacher_pairs, teacher, INFERENCE_CONFIG)
    teacher_pairs['teacher_logit'] = teacher_logits.astype(np.float32)
    teacher_pairs['teacher_probability'] = (1 / (1 + np.exp(-np.clip(teacher_logits, -30, 30)))).astype(np.float32)
    teacher_pairs.to_parquet(WORK_DIR / 'teacher_soft_labels.parquet', index=False)
    print({'teacher_rows': len(teacher_pairs), 'used_batch': used_batch})
else:
    print('Teacher kapalı; 0.6B/BGE OOF tamamlanmadan 4B maliyetine girilmiyor.')

## 10. 4.096-pair benchmark ve chunked test inference

Her shard ordered-ID hash, model fingerprint, satır sınırı ve raw logit taşır. OOM aynı aralıkta batch yarıya indirilerek tekrar denenir. Query satırlarının contiguous olduğu varsayılmaz; `_row_idx` finalde özgün sırayı korur.

In [ ]:
from trendyol_v3_core import AuditConfig, load_data_bundle
from trendyol_v3_reranker import benchmark_inference, chunked_pair_inference

RUN_TEST_INFERENCE = False
FULL_BUNDLE = None
if RUN_TEST_INFERENCE:
    FULL_BUNDLE = load_data_bundle(DATA_DIR, AuditConfig(debug=False, seed=SEED))
    for model_key, spec in MODEL_SPECS.items():
        for seed in spec['seeds']:
            for fold in range(N_SPLITS):
                checkpoint = MODEL_RUN_DIR / model_key / f'seed_{seed}' / f'fold_{fold}' / 'final'
                if not checkpoint.exists():
                    raise FileNotFoundError(f'Eksik trained checkpoint: {checkpoint}')
                output_dir = WORK_DIR / 'test_scores' / model_key / f'seed_{seed}' / f'fold_{fold}'
                path = chunked_pair_inference(
                    DATA_DIR / 'submission_pairs.csv', FULL_BUNDLE.terms, FULL_BUNDLE.items,
                    checkpoint, output_dir, spec['config'], INFERENCE_CONFIG,
                    model_key=f'{model_key}_seed{seed}_fold{fold}',
                    item_view_cache_path=WORK_DIR / 'cache' / f"item_view_{spec['config'].product_view}.parquet",
                )
                print(path)
else:
    print('Test inference kapalı. Önce OOF model seçimi ve 4.096-pair süre benchmarkı tamamlanmalı.')

## 11. OOF ensemble, calibration ve threshold

Average, constrained weighted average, query-local rank, geometric mean ve logistic stacking fold-dışı OOF ile karşılaştırılır. Seçili ensemble yeniden meta cross-fit ile temperature/Platt/isotonic/beta kalibre edilir. Public LB ağırlık veya threshold fit verisi değildir.

In [ ]:
from trendyol_v3_ensemble import (
    align_model_scores, calibrate_final_ensemble, cross_fit_ensembles,
    fit_full_ensemble_and_predict, score_correlation_matrix,
)
from trendyol_v3_validation import cross_fit_calibration_and_threshold

RUN_ENSEMBLE = False
OOF_SCORE_FILES: dict[str, Path] = {}
TEST_SCORE_FILES: dict[str, Path] = {}
if RUN_ENSEMBLE:
    if set(OOF_SCORE_FILES) != set(TEST_SCORE_FILES) or len(OOF_SCORE_FILES) < 2:
        raise ValueError('Ensemble için aynı model key kümesinde en az iki OOF/test score dosyası gerekir')
    aligned_oof = align_model_scores(OOF_SCORE_FILES, oof=True)
    aligned_test = align_model_scores(TEST_SCORE_FILES, oof=False)
    score_correlation_matrix(aligned_oof).to_csv(WORK_DIR / 'oof_correlations.csv')
    ensemble_oof, ensemble_report = cross_fit_ensembles(aligned_oof, seed=SEED)
    display(ensemble_report)
    selected_method = str(ensemble_report.iloc[0].ensemble_method)
    selected_oof = ensemble_oof[ensemble_oof.ensemble_method.eq(selected_method)].copy()
    calibrated_oof, calibration_report = cross_fit_calibration_and_threshold(
        selected_oof, methods=('none', 'temperature', 'platt', 'isotonic', 'beta'), seed=SEED
    )
    display(calibration_report)
    selected_calibration = str(calibration_report.iloc[0].calibration_method)
    test_raw, parameters = fit_full_ensemble_and_predict(aligned_oof, aligned_test, selected_method, seed=SEED)
    final_scores, FINAL_THRESHOLD, _ = calibrate_final_ensemble(
        selected_oof, test_raw, method=selected_calibration, seed=SEED
    )
    FINAL_SCORES_PATH = WORK_DIR / 'ensemble_test_probabilities.parquet'
    final_scores.to_parquet(FINAL_SCORES_PATH, index=False)
    (WORK_DIR / 'ensemble_parameters.json').write_text(json.dumps({
        'method': selected_method, 'calibration': selected_calibration,
        'threshold': FINAL_THRESHOLD, **parameters,
    }, indent=2))
else:
    FINAL_THRESHOLD = None
    FINAL_SCORES_PATH = None
    print('Ensemble kapalı; OOF_SCORE_FILES ve TEST_SCORE_FILES model artefaktları oluşunca açılır.')

## 12. Final submission ve deney/ablation logu

Final hücre sample ID sırasını, satır sayısını, uniqueness'i, eksik score'u ve binary prediction'ı doğrular. Binary CSV yanında raw/calibrated probability Parquet ve ilk/son satır summary JSON kalır. Query başına en az bir pozitif/top-k yalnız meta cross-fit OOF artışı gösterildiyse açılır.

In [ ]:
from trendyol_v3_ensemble import ExperimentLogger, ablation_table, validate_and_write_submission

RUN_SUBMISSION = False
if RUN_SUBMISSION:
    if FINAL_SCORES_PATH is None or FINAL_THRESHOLD is None:
        raise RuntimeError('Final ensemble probability ve OOF-selected threshold oluşmadı')
    final_scores = pd.read_parquet(FINAL_SCORES_PATH)
    SUMMARY = validate_and_write_submission(
        final_scores, DATA_DIR / 'sample_submission.csv', DATA_DIR / 'submission_pairs.csv',
        WORK_DIR / 'final_submission.csv', threshold=float(FINAL_THRESHOLD),
        ensure_one_per_query=False, top_k=None, contradiction_penalty=0.0,
    )
    display(pd.DataFrame(SUMMARY['first_rows']))
    display(pd.DataFrame(SUMMARY['last_rows']))
else:
    print('Submission kapalı; yalnız tam OOF ensemble ve meta cross-fit threshold sonrası açılır.')

ABLATION_COMPONENTS = [
    'old_negatives', 'false_negative_filter', 'dynamic_hard_negatives',
    'field_tagged_text', 'query_normalization', 'pairwise_or_listwise',
    'qwen3_reranker', 'second_reranker', 'teacher_distillation',
    'calibration', 'group_postprocessing', 'ensemble',
]
print('Zorunlu ablation sırası:', ABLATION_COMPONENTS)